# Phase 8 — Schema Intelligence

This notebook builds schema intelligence for the
Databricks GenAI Data Analyst Copilot.

The profiler discovers tables and columns from Unity Catalog,
then generates metadata describing:

- table name
- column name
- data type
- nullable
- distinct count
- sample values
- minimum value
- maximum value
- average value where applicable
- column description

The resulting metadata is stored in:

genai_copilot.gold.schema_metadata

This metadata will later be used by the Natural Language
to SQL component to select relevant tables and columns.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
    BooleanType,
    TimestampType,
)

from datetime import datetime

In [0]:
CATALOG = "genai_copilot"
GOLD_SCHEMA = "gold"
METADATA_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.schema_metadata"

print("Catalog:", CATALOG)
print("Metadata table:", METADATA_TABLE)

In [0]:
tables_df = spark.sql(f"""
SELECT
    table_catalog,
    table_schema,
    table_name,
    table_type,
    comment
FROM {CATALOG}.information_schema.tables
WHERE table_schema IN (
    'bronze',
    'silver',
    'gold'
)
ORDER BY table_schema, table_name
""")

display(tables_df)

In [0]:
table_list = [
    row["table_name"]
    for row in tables_df.collect()
]

print("Discovered tables:")
for table_name in table_list:
    print("-", table_name)

In [0]:
columns_df = spark.sql(f"""
SELECT
    table_catalog,
    table_schema,
    table_name,
    column_name,
    ordinal_position,
    data_type,
    is_nullable,
    comment
FROM {CATALOG}.information_schema.columns
WHERE table_schema IN (
    'bronze',
    'silver',
    'gold'
)
ORDER BY
    table_schema,
    table_name,
    ordinal_position
""")

display(columns_df)

In [0]:
column_count = columns_df.count()

print(f"Total columns discovered: {column_count}")

In [0]:
PROFILE_TABLES = [
    f"{CATALOG}.silver.sales",
    f"{CATALOG}.gold.monthly_sales",
    f"{CATALOG}.gold.region_sales",
    f"{CATALOG}.gold.customer_metrics",
    f"{CATALOG}.gold.product_metrics",
    f"{CATALOG}.gold.category_metrics",
]

print("Tables selected for profiling:")

for table_name in PROFILE_TABLES:
    print("-", table_name)

In [0]:
existing_tables = set()

for row in tables_df.collect():
    full_name = (
        f"{row['table_catalog']}."
        f"{row['table_schema']}."
        f"{row['table_name']}"
    )

    existing_tables.add(full_name)

available_profile_tables = [
    table_name
    for table_name in PROFILE_TABLES
    if table_name in existing_tables
]

print("Available tables for profiling:")

for table_name in available_profile_tables:
    print("-", table_name)

print()
print(
    f"{len(available_profile_tables)} "
    f"of {len(PROFILE_TABLES)} requested tables are available."
)

In [0]:
metadata_results = []

profile_timestamp = datetime.now()

print("Profiling started:", profile_timestamp)

In [0]:
for table_name in available_profile_tables:

    print(f"\nProfiling table: {table_name}")

    table_df = spark.table(table_name)

    table_columns = table_df.columns

    column_metadata = columns_df.filter(
        F.concat_ws(
            ".",
            F.col("table_catalog"),
            F.col("table_schema"),
            F.col("table_name")
        ) == table_name
    ).collect()

    for column_info in column_metadata:

        column_name = column_info["column_name"]
        data_type = column_info["data_type"]
        nullable = column_info["is_nullable"] == "YES"
        description = column_info["comment"]

        column = F.col(column_name)

        distinct_count = (
            table_df
            .select(column)
            .distinct()
            .count()
        )

        sample_rows = (
            table_df
            .select(column)
            .filter(column.isNotNull())
            .limit(5)
            .collect()
        )

        sample_values = [
            str(row[0])
            for row in sample_rows
        ]

        min_value = None
        max_value = None
        avg_value = None

        if data_type in [
            "INT",
            "BIGINT",
            "DOUBLE",
            "FLOAT",
            "DECIMAL",
            "SMALLINT",
            "TINYINT",
        ]:

            stats = table_df.select(
                F.min(column).alias("min_value"),
                F.max(column).alias("max_value"),
                F.avg(column).alias("avg_value")
            ).collect()[0]

            min_value = (
                float(stats["min_value"])
                if stats["min_value"] is not None
                else None
            )

            max_value = (
                float(stats["max_value"])
                if stats["max_value"] is not None
                else None
            )

            avg_value = (
                float(stats["avg_value"])
                if stats["avg_value"] is not None
                else None
            )

        metadata_results.append({
            "table_name": table_name,
            "column_name": column_name,
            "data_type": data_type,
            "nullable": nullable,
            "distinct_count": distinct_count,
            "sample_values": ", ".join(sample_values),
            "min_value": min_value,
            "max_value": max_value,
            "avg_value": avg_value,
            "description": description,
            "profile_timestamp": profile_timestamp,
        })

print(
    f"\nGenerated metadata for "
    f"{len(metadata_results)} columns."
)

In [0]:
metadata_schema = StructType([
    StructField("table_name", StringType(), False),
    StructField("column_name", StringType(), False),
    StructField("data_type", StringType(), False),
    StructField("nullable", BooleanType(), False),
    StructField("distinct_count", LongType(), False),
    StructField("sample_values", StringType(), True),
    StructField("min_value", DoubleType(), True),
    StructField("max_value", DoubleType(), True),
    StructField("avg_value", DoubleType(), True),
    StructField("description", StringType(), True),
    StructField("profile_timestamp", TimestampType(), False),
])

metadata_df = spark.createDataFrame(
    metadata_results,
    schema=metadata_schema
)

display(metadata_df)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {METADATA_TABLE} (
    table_name STRING,
    column_name STRING,
    data_type STRING,
    nullable BOOLEAN,
    distinct_count BIGINT,
    sample_values STRING,
    min_value DOUBLE,
    max_value DOUBLE,
    avg_value DOUBLE,
    description STRING,
    profile_timestamp TIMESTAMP
)
USING DELTA
""")

print("Verified:", METADATA_TABLE)

In [0]:
(
    metadata_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(METADATA_TABLE)
)

print(
    f"Schema metadata successfully written to "
    f"{METADATA_TABLE}"
)

In [0]:
saved_metadata = spark.table(METADATA_TABLE)

print(
    "Metadata rows:",
    saved_metadata.count()
)

display(
    saved_metadata
    .orderBy(
        "table_name",
        "column_name"
    )
)

In [0]:
table_summary = (
    saved_metadata
    .groupBy("table_name")
    .agg(
        F.count("*").alias("column_count")
    )
    .orderBy("table_name")
)

display(table_summary)

In [0]:
schema_rows = (
    saved_metadata
    .select(
        "table_name",
        "column_name",
        "data_type",
        "description"
    )
    .orderBy(
        "table_name",
        "column_name"
    )
    .collect()
)

schema_context = {}

for row in schema_rows:

    table_name = row["table_name"]

    if table_name not in schema_context:
        schema_context[table_name] = []

    schema_context[table_name].append({
        "column": row["column_name"],
        "type": row["data_type"],
        "description": row["description"]
    })

for table_name, columns in schema_context.items():

    print(f"\nTABLE: {table_name}")

    for column in columns:
        print(
            f"  - {column['column']} "
            f"({column['type']})"
        )

In [0]:
business_descriptions = {
    "order_id": "Unique identifier for a customer order.",
    "order_date": "Date on which the order was placed.",
    "customer_id": "Unique identifier for the customer.",
    "customer_name": "Synthetic customer name associated with the order.",
    "region": "Geographical sales region.",
    "country": "Country where the order was generated.",
    "product_id": "Unique identifier for the product.",
    "product_name": "Name of the product purchased.",
    "category": "Product category.",
    "quantity": "Number of units purchased.",
    "unit_price": "Price of one unit before discount.",
    "discount": "Discount applied to the order, represented as a decimal percentage.",
    "revenue": "Revenue generated by the order after applicable discount.",
    "cost": "Cost associated with fulfilling the order.",
    "profit": "Revenue minus cost.",
    "profit_margin": "Profit divided by revenue.",
    "sales_channel": "Channel through which the order was sold.",
    "order_status": "Current status of the order."
}

print(
    f"Business descriptions defined for "
    f"{len(business_descriptions)} columns."
)

In [0]:
description_expr = F.col("description")

for column_name, description in business_descriptions.items():

    description_expr = F.when(
        F.col("column_name") == column_name,
        F.lit(description)
    ).otherwise(description_expr)

enriched_metadata_df = (
    saved_metadata
    .withColumn(
        "description",
        description_expr
    )
)

display(
    enriched_metadata_df
    .filter(
        F.col("table_name") == f"{CATALOG}.silver.sales"
    )
    .select(
        "table_name",
        "column_name",
        "data_type",
        "description"
    )
    .orderBy("column_name")
)

In [0]:
(
    enriched_metadata_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(METADATA_TABLE)
)

print("Enriched schema metadata saved.")

In [0]:
final_metadata = spark.table(METADATA_TABLE)

metadata_count = final_metadata.count()

print("=" * 60)
print("SCHEMA INTELLIGENCE VALIDATION")
print("=" * 60)

print("Metadata table:", METADATA_TABLE)
print("Metadata rows:", metadata_count)

print()

if metadata_count > 0:
    print("STATUS: PASS")
    print("Schema metadata is ready for GenAI.")
else:
    print("STATUS: FAIL")
    print("No schema metadata was generated.")

In [0]:
display(
    final_metadata
    .select(
        "table_name",
        "column_name",
        "data_type",
        "distinct_count",
        "sample_values",
        "min_value",
        "max_value",
        "description"
    )
    .orderBy(
        "table_name",
        "column_name"
    )
    .limit(100)
)

8.26 What this phase demonstrates

This phase is much more important than it might initially look.

You are demonstrating:

Data Engineering

You are programmatically inspecting your lakehouse.

Unity Catalog

You're using catalog metadata rather than hard-coded table definitions.

PySpark

You're calculating statistics and profiling columns.

Delta Lake

You're persisting metadata into:

genai_copilot.gold.schema_metadata
GenAI architecture

Most importantly:

                    ┌─────────────────┐
                    │ Unity Catalog   │
                    └────────┬────────┘
                             ↓
                    ┌─────────────────┐
                    │ Schema Profiler │
                    └────────┬────────┘
                             ↓
                 ┌──────────────────────┐
                 │ schema_metadata      │
                 └──────────┬───────────┘
                            ↓
                  Relevant schema only
                            ↓
                         LLM
                            ↓
                       SQL generation

That is much closer to a production-oriented Text-to-SQL system than simply asking an LLM:

"Here is my database, write SQL."

In [0]:
%sql
SELECT
    table_name,
    COUNT(*) AS column_count
FROM genai_copilot.gold.schema_metadata
GROUP BY table_name
ORDER BY table_name;

In [0]:
%sql
SELECT
    table_name,
    column_name,
    data_type,
    distinct_count,
    description
FROM genai_copilot.gold.schema_metadata
WHERE table_name = 'genai_copilot.silver.sales'
ORDER BY column_name;